# Phase 1: Data Cleaning & Filtering

**Goal:** Loading all raw data, filter London.

**Outputs saved to:** `outputs/phase1/`
- `phase1_crimes_london.parquet` : all street crimes for Met + City of London
- `phase1_outcomes_london.parquet`: all outcomes for Met + City of London
- `phase1_stop_search_london.parquet`: all stop & searches for Met + City of London
- `phase1_footfall_monthly.parquet`: TfL footfall aggregated to monthly per station
- `phase1_temperature_monthly.parquet`: monthly mean temperature for London (Apr 2023 – Mar 2026)

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import xarray as xr
from pathlib import Path

In [ ]:
# Loading dataset 
BASE       = Path("Dataset")
OUT        = BASE / 'outputs' / 'phase1'
OUT.mkdir(parents=True, exist_ok=True)

LONDON_FORCES = {'Metropolitan Police Service', 'City of London Police'}

# Crime archive folders in chronological order
ARCHIVES = ['0423-0424', '0524-0525', '0625-0326']

# London grid slice indices (pre-computed from lat/lon mask)
LON_Y_SLICE = slice(355, 402)
LON_X_SLICE = slice(703, 762)

print('Output folder:', OUT)
print('London forces:', LONDON_FORCES)

Output folder: C:\Users\mbeck\OneDrive\Documents\CBL-16_data\outputs\phase1
London forces: {'Metropolitan Police Service', 'City of London Police'}


## Section-1: Crime Data
Load street crimes, outcomes, and stop & search across all 36 months. Filter to London forces only.

In [3]:
# --- PHASE 1A: Street crimes ---
print('Loading street crime files...')
street_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'crimes' / archive
    files = sorted(archive_path.rglob('*-street.csv'))
    print(f'  {archive}: {len(files)} files')
    for f in files:
        try:
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = df[df['Falls within'].isin(LONDON_FORCES)]
            if len(df) > 0:
                street_chunks.append(df)
        except Exception as e:
            print(f'  [SKIP] {f.name}: {e}')

crimes = pd.concat(street_chunks, ignore_index=True)
crimes.columns = crimes.columns.str.strip()
crimes['Month'] = pd.to_datetime(crimes['Month'], format='%Y-%m')

print(f'\nShape: {crimes.shape}')
print(f'Date range: {crimes["Month"].min().strftime("%b %Y")} → {crimes["Month"].max().strftime("%b %Y")}')
print(f'Forces: {crimes["Falls within"].unique()}')
print(f'Nulls:\n{crimes.isnull().sum()}')
crimes.head(3)

Loading street crime files...
  0423-0424: 572 files
  0524-0525: 568 files
  0625-0326: 421 files

Shape: (3443915, 12)
Date range: Apr 2023 → Mar 2026
Forces: ['City of London Police' 'Metropolitan Police Service']
Nulls:
Crime ID                  696507
Month                          0
Reported by                    0
Falls within                   0
Longitude                  18691
Latitude                   18691
Location                       0
LSOA code                  18692
LSOA name                  18692
Crime type                     0
Last outcome category     696507
Context                  3443915
dtype: int64


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,41f5b4e5c183ea17cc6c92fecbd2f408094104d0d0dc0e...,2023-04-01,City of London Police,City of London Police,-0.110350,51.518090,On or near Holborn,E01000917,Camden 027C,Other theft,Status update unavailable,NaN
1,ed18a10886a5a4c6180ae489844993780da0f3090247ed...,2023-04-01,City of London Police,City of London Police,-0.110350,51.518090,On or near Holborn,E01000917,Camden 027C,Other theft,Investigation complete; no suspect identified,NaN
2,b8e577f2767ed58fe027b3f7e2eeaf87967f37abebda90...,2023-04-01,City of London Police,City of London Police,-0.110350,51.518090,On or near Holborn,E01000917,Camden 027C,Theft from the person,Investigation complete; no suspect identified,NaN


In [ ]:
# PHASE 1B: Outcomes
print('Loading outcomes files...')
outcome_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'crimes' / archive
    files = sorted(archive_path.rglob('*-outcomes.csv'))
    print(f'  {archive}: {len(files)} files')
    for f in files:
        try:
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = df[df['Falls within'].isin(LONDON_FORCES)]
            if len(df) > 0:
                outcome_chunks.append(df)
        except Exception as e:
            print(f'  [SKIP] {f.name}: {e}')

outcomes = pd.concat(outcome_chunks, ignore_index=True)
outcomes.columns = outcomes.columns.str.strip()
outcomes['Month'] = pd.to_datetime(outcomes['Month'], format='%Y-%m')

print(f'\nShape: {outcomes.shape}')
print(f'Date range: {outcomes["Month"].min().strftime("%b %Y")} → {outcomes["Month"].max().strftime("%b %Y")}')
print(f'Nulls:\n{outcomes.isnull().sum()}')
outcomes.head(3)

Loading outcomes files...
  0423-0424: 546 files
  0524-0525: 545 files
  0625-0326: 411 files

Shape: (2905468, 10)
Date range: Apr 2023 → Mar 2026
Nulls:
Crime ID            0
Month               0
Reported by         0
Falls within        0
Longitude       19136
Latitude        19136
Location            0
LSOA code       19137
LSOA name       19137
Outcome type        0
dtype: int64


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Outcome type
0,c5bc4aa811362bae948e311288e5413a1d47cae8ebb9ec...,2023-04-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Offender given a drugs possession warning
1,716e353875fbeffc7b01fb45e658b9d6a7bf6e1f03a9d8...,2023-04-01,City of London Police,City of London Police,NaN,NaN,No location,NaN,NaN,Investigation complete; no suspect identified
2,b0bffb2ae0f2d3d4268c22368747772d1cf2c7a796644a...,2023-04-01,City of London Police,City of London Police,-0.095924,51.513882,On or near,E01032739,City of London 001F,Investigation complete; no suspect identified


In [ ]:
# PHASE 1C: Stop & Search
print('Loading stop & search files...')
ss_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'crimes' / archive
    # Metropolitan and City of London stop & search files only
    for pattern in ['*-metropolitan-stop-and-search.csv', '*-city-of-london-stop-and-search.csv']:
        files = sorted(archive_path.rglob(pattern))
        print(f'  {archive} / {pattern.split("-")[1]}: {len(files)} files')
        for f in files:
            try:
                df = pd.read_csv(f, dtype=str, low_memory=False)
                if len(df) > 0:
                    # Tag which force this came from using the filename
                    force = 'Metropolitan Police Service' if 'metropolitan' in f.name else 'City of London Police'
                    df['Falls within'] = force
                    ss_chunks.append(df)
            except Exception as e:
                print(f'  [SKIP] {f.name}: {e}')

stop_search = pd.concat(ss_chunks, ignore_index=True)
stop_search.columns = stop_search.columns.str.strip()
stop_search['Date'] = pd.to_datetime(stop_search['Date'], utc=True, errors='coerce')
stop_search['Month'] = stop_search['Date'].dt.to_period('M').dt.to_timestamp()

# Drop rows with no coordinates (can't spatially join later)
before = len(stop_search)
stop_search = stop_search.dropna(subset=['Latitude', 'Longitude'])
stop_search['Latitude']  = stop_search['Latitude'].astype(float)
stop_search['Longitude'] = stop_search['Longitude'].astype(float)
print(f'\nDropped {before - len(stop_search)} rows with missing coordinates')

print(f'Shape: {stop_search.shape}')
print(f'Date range: {stop_search["Month"].min().strftime("%b %Y")} → {stop_search["Month"].max().strftime("%b %Y")}')
print(f'Nulls:\n{stop_search.isnull().sum()}')
stop_search.head(3)

Loading stop & search files...
  0423-0424 / metropolitan: 13 files
  0423-0424 / city: 13 files
  0524-0525 / metropolitan: 13 files
  0524-0525 / city: 13 files
  0625-0326 / metropolitan: 9 files
  0625-0326 / city: 10 files


C:\Users\mbeck\AppData\Local\Temp\ipykernel_26796\3108624541.py:25: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  stop_search['Month'] = stop_search['Date'].dt.to_period('M').dt.to_timestamp()



Dropped 19729 rows with missing coordinates
Shape: (372018, 17)
Date range: Mar 2023 → Mar 2026
Nulls:
Type                                             0
Date                                             0
Part of a policing operation                  5063
Policing operation                          372018
Latitude                                         0
Longitude                                        0
Gender                                       15573
Age range                                    46442
Self-defined ethnicity                        5576
Officer-defined ethnicity                    18784
Legislation                                    639
Object of search                              1225
Outcome                                         25
Outcome linked to object of search          103206
Removal of more than just outer clothing    103206
Falls within                                     0
Month                                            0
dtype: int64


,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,Legislation,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing,Falls within,Month
0,Person search,2023-03-31 23:01:00+00:00,False,NaN,51.516275,-0.126121,Male,18-24,Black/African/Caribbean/Black British - African,Black,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,NaN,NaN,Metropolitan Police Service,2023-03-01
1,Person and Vehicle search,2023-03-31 23:05:00+00:00,False,NaN,51.550226,0.002608,Female,18-24,Other ethnic group - Any other ethnic group,Other,Misuse of Drugs Act 1971 (section 23),Controlled drugs,Community resolution,NaN,NaN,Metropolitan Police Service,2023-03-01
2,Person and Vehicle search,2023-03-31 23:05:00+00:00,False,NaN,51.550226,0.002608,Female,18-24,Other ethnic group - Any other ethnic group,Other,Misuse of Drugs Act 1971 (section 23),Controlled drugs,Arrest,NaN,NaN,Metropolitan Police Service,2023-03-01


## Section-2: TfL Footfall
Three CSVs (2023, 2024, 2025-26) with daily station-level entry/exit counts. Aggregate to monthly totals per station.

In [6]:
# --- PHASE 1D: Footfall ---
footfall_files = [
    BASE / 'footfall' / 'StationFootfall_2023.csv',
    BASE / 'footfall' / 'StationFootfall_2024.csv',
    BASE / 'footfall' / 'StationFootfall_2025_2026 .csv',
]

print('Loading footfall files...')
ff_chunks = []
for f in footfall_files:
    df = pd.read_csv(f, dtype={'TravelDate': str})
    df.columns = df.columns.str.strip()
    ff_chunks.append(df)
    print(f'  {f.name}: {df.shape}')

footfall_raw = pd.concat(ff_chunks, ignore_index=True)
footfall_raw['TravelDate'] = pd.to_datetime(footfall_raw['TravelDate'], format='%Y%m%d')
footfall_raw['Month'] = footfall_raw['TravelDate'].dt.to_period('M').dt.to_timestamp()
footfall_raw['TotalFootfall'] = footfall_raw['EntryTapCount'] + footfall_raw['ExitTapCount']

# Aggregate daily → monthly per station
footfall_monthly = (
    footfall_raw
    .groupby(['Month', 'Station'], as_index=False)
    .agg(TotalFootfall=('TotalFootfall', 'sum'))
)

# Filter to project period: Apr 2023 – Mar 2026
footfall_monthly = footfall_monthly[
    (footfall_monthly['Month'] >= '2023-04') &
    (footfall_monthly['Month'] <= '2026-03')
].copy()

print(f'\nShape (monthly): {footfall_monthly.shape}')
print(f'Date range: {footfall_monthly["Month"].min().strftime("%b %Y")} → {footfall_monthly["Month"].max().strftime("%b %Y")}')
print(f'Unique stations: {footfall_monthly["Station"].nunique()}')
print(f'Nulls: {footfall_monthly.isnull().sum().to_dict()}')
footfall_monthly.head(3)

Loading footfall files...
  StationFootfall_2023.csv: (156491, 5)
  StationFootfall_2024.csv: (156995, 5)
  StationFootfall_2025_2026 .csv: (205524, 5)

Shape (monthly): (15615, 3)
Date range: Apr 2023 → Mar 2026
Unique stations: 437
Nulls: {'Month': 0, 'Station': 0, 'TotalFootfall': 0}


,Month,Station,TotalFootfall
1302,2023-04-01,Abbey Road DLR,43013
1303,2023-04-01,Abbey Wood,616258
1304,2023-04-01,Acton Central,114584


## Section-3: Temperature (HadUK-Grid NetCDF)

Extract monthly mean temperature for London from 1km UK grid.
- London grid slice: y-index 355–401, x-index 703–761 (pre-verified via lat/lon mask)
- Mean temperature = (tasmax + tasmin) / 2, averaged over all London grid cells

In [ ]:
def extract_london_monthly(nc_file, var_name):
    """Open a NetCDF file and return a Series of monthly mean values for London."""
    ds = xr.open_dataset(nc_file)
    london = ds[var_name].isel(
        projection_y_coordinate=LON_Y_SLICE,
        projection_x_coordinate=LON_X_SLICE
    )
    # Mean over all London grid cells for each time step
    monthly_vals = london.mean(dim=['projection_y_coordinate', 'projection_x_coordinate'])
    times  = pd.to_datetime(ds.time.values)
    ds.close()
    return pd.Series(monthly_vals.values, index=times)

TEMP_DIR = BASE / 'temp-london'

# tasmax
print('Extracting tasmax...')
tasmax_parts = []

# Annual files
for year in ['202301-202312', '202401-202412']:
    f = TEMP_DIR / f'tasmax_hadukgrid_uk_1km_mon_{year}.nc'
    s = extract_london_monthly(f, 'tasmax')
    tasmax_parts.append(s)
    print(f'  {f.name}: {len(s)} months')

# Individual monthly files (2025-01 through 2026-03)
for ym in ['202501','202502','202503','202504','202505','202506',
           '202507','202508','202509','202510','202511','202512',
           '202601','202602','202603']:
    f = TEMP_DIR / f'tasmax_hadukgrid_uk_1km_mon_{ym}.nc'
    if f.exists():
        s = extract_london_monthly(f, 'tasmax')
        tasmax_parts.append(s)

tasmax_series = pd.concat(tasmax_parts).sort_index()
print(f'Total tasmax months: {len(tasmax_series)}')

# tasmin
print('\nExtracting tasmin...')
tasmin_parts = []

for year in ['202301-202312', '202401-202412']:
    f = TEMP_DIR / f'tasmin_hadukgrid_uk_1km_mon_{year}.nc'
    s = extract_london_monthly(f, 'tasmin')
    tasmin_parts.append(s)
    print(f'  {f.name}: {len(s)} months')

for ym in ['202501','202502','202503','202504','202505','202506',
           '202507','202508','202509','202510','202511','202512',
           '202601','202602','202603']:
    f = TEMP_DIR / f'tasmin_hadukgrid_uk_1km_mon_{ym}.nc'
    if f.exists():
        s = extract_london_monthly(f, 'tasmin')
        tasmin_parts.append(s)

tasmin_series = pd.concat(tasmin_parts).sort_index()
print(f'Total tasmin months: {len(tasmin_series)}')

Extracting tasmax...
  tasmax_hadukgrid_uk_1km_mon_202301-202312.nc: 12 months
  tasmax_hadukgrid_uk_1km_mon_202401-202412.nc: 12 months
Total tasmax months: 39

Extracting tasmin...
  tasmin_hadukgrid_uk_1km_mon_202301-202312.nc: 12 months
  tasmin_hadukgrid_uk_1km_mon_202401-202412.nc: 12 months
Total tasmin months: 39


In [8]:
# Combine into a DataFrame and filter to project period
temp_df = pd.DataFrame({
    'tasmax': tasmax_series,
    'tasmin': tasmin_series
})
temp_df.index.name = 'Month'
temp_df['avg_temperature'] = (temp_df['tasmax'] + temp_df['tasmin']) / 2
temp_df = temp_df.reset_index()
temp_df['Month'] = pd.to_datetime(temp_df['Month']).dt.to_period('M').dt.to_timestamp()

# Filter to Apr 2023 – Mar 2026
temp_df = temp_df[
    (temp_df['Month'] >= '2023-04') &
    (temp_df['Month'] <= '2026-03')
].reset_index(drop=True)

print(f'Shape: {temp_df.shape}')
print(f'Date range: {temp_df["Month"].min().strftime("%b %Y")} → {temp_df["Month"].max().strftime("%b %Y")}')
print(f'\nTemperature stats (°C):')
print(temp_df[['tasmax', 'tasmin', 'avg_temperature']].describe().round(2))
temp_df.head(6)

Shape: (36, 4)
Date range: Apr 2023 → Mar 2026

Temperature stats (°C):
       tasmax  tasmin  avg_temperature
count   36.00   36.00            36.00
mean    16.15    7.84            12.00
std      5.69    3.96             4.77
min      6.93    0.76             3.85
25%     11.20    4.90             8.08
50%     15.64    7.38            11.81
75%     21.23   11.44            15.94
max     24.98   14.47            19.72


,Month,tasmax,tasmin,avg_temperature
0,2023-04-01,13.968946,4.861785,9.415366
1,2023-05-01,17.903289,8.135663,13.019476
2,2023-06-01,24.314618,12.277098,18.295858
3,2023-07-01,22.176245,13.160191,17.668218
4,2023-08-01,22.391694,12.803888,17.597791
5,2023-09-01,23.633742,13.426316,18.530029


## Section-4: Save Outputs

In [ ]:
# Save all outputs as parquet(Outputs)
saves = {
    'phase1_crimes_london.parquet':      crimes,
    'phase1_outcomes_london.parquet':    outcomes,
    'phase1_stop_search_london.parquet': stop_search,
    'phase1_footfall_monthly.parquet':   footfall_monthly,
    'phase1_temperature_monthly.parquet': temp_df,
}

for filename, df in saves.items():
    path = OUT / filename
    df.to_parquet(path, index=False)
    size_mb = path.stat().st_size / 1e6
    print(f'  Saved {filename} — {df.shape[0]:,} rows x {df.shape[1]} cols ({size_mb:.1f} MB)')

print('\nPhase 1 complete.')

  Saved phase1_crimes_london.parquet — 3,443,915 rows x 12 cols (211.7 MB)
  Saved phase1_outcomes_london.parquet — 2,905,468 rows x 10 cols (214.9 MB)
  Saved phase1_stop_search_london.parquet — 372,018 rows x 17 cols (5.5 MB)
  Saved phase1_footfall_monthly.parquet — 15,615 rows x 3 cols (0.1 MB)
  Saved phase1_temperature_monthly.parquet — 36 rows x 4 cols (0.0 MB)

Phase 1 complete.


In [ ]:
# Phase 1 summary
print('=' * 55)
print('PHASE 1 SUMMARY')
print('=' * 55)
print(f'Street crimes (London):   {len(crimes):>10,} rows')
print(f'Outcomes (London):        {len(outcomes):>10,} rows')
print(f'Stop & search (London):   {len(stop_search):>10,} rows')
print(f'Footfall (monthly):       {len(footfall_monthly):>10,} rows  ({footfall_monthly["Station"].nunique()} stations)')
print(f'Temperature (monthly):    {len(temp_df):>10,} rows')
print('=' * 55)
print(f'Outputs saved to: {OUT}')

PHASE 1 SUMMARY
Street crimes (London):    3,443,915 rows
Outcomes (London):         2,905,468 rows
Stop & search (London):      372,018 rows
Footfall (monthly):           15,615 rows  (437 stations)
Temperature (monthly):            36 rows
Outputs saved to: C:\Users\mbeck\OneDrive\Documents\CBL-16_data\outputs\phase1
